In [ ]:
from jax import random
import json
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import pycountry

from emu_renewal.inputs import DATA_PATH, get_standard_priors, get_country_vacc_data, \
    get_country_vars, find_relevant_vars, get_worldbank_national_pop, get_country_mobility, \
    get_standard_targets, get_row_proportions, get_indicator_series_from_who_data
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import store_outputs
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
proc_update_freq = 7
init_duration = 50
seed_duration = 10
iterations = 100
priors = get_standard_priors()
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
for country in ["Czechia"]:

    # Get basic data
    print(f"Running {country} analyses")
    analysis_time = datetime.now().strftime("%Y%m%d_%H%M")
    iso2 = pycountry.countries.lookup(country).alpha_2
    iso3 = pycountry.countries.lookup(country).alpha_3
    country_name = pycountry.countries.lookup(country).name
    pop = get_worldbank_national_pop(iso3)

    # Start and end times
    deaths_series = get_indicator_series_from_who_data("New_deaths", iso3)
    per_capita_deaths = deaths_series / pop
    death_start_threshold = 2e-6
    start_time = per_capita_deaths.index[per_capita_deaths.gt(death_start_threshold)].min()
    vacc_data = get_country_vacc_data(country)
    cov_threshold = 0.05
    end_time = vacc_data[vacc_data.gt(cov_threshold * 100)].idxmin()

    # Targets
    hosp_indicator, hosp_colour, hosp_output = ("Weekly new hospital admissions", "black", "weekly_admissions") if iso2 in initial_countries["admissions"] else ("Daily hospital occupancy", "red", "occupancy")
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(iso2, start_time, end_time, 50, hosp_indicator)
    cases_target = cases_target[cases_target.index >= datetime(2020, 6, 1)]  # Ignore initial cases before testing scaled up
    min_var_samples = 10
    var_country_name = pycountry.countries.lookup(iso3).official_name if iso3 in ["CZE"] else pycountry.countries.lookup(iso3).name
    var_data = get_country_vars(var_country_name)
    var_data = var_data[var_data.sum(axis=1) >= min_var_samples]
    var_props = get_row_proportions(var_data)
    prealpha_vars = ["20A.EU1"] if iso3 == "LTU" else ["20A.EU1", "20A.EU2"]  # Lithuania has no 20A.EU2
    prealpha_prop = var_data[prealpha_vars].sum(axis=1) / var_data.sum(axis=1)
    min_var_prop = 0.05
    pre_alpha_prop = prealpha_prop[(min_var_prop < prealpha_prop) & (prealpha_prop < 1.0 - min_var_prop)]
    
    # Collate targets
    seroprev_target_dict = {"seropos": StandardDispTarget(seroprev_target, weight=10.0)} if any(seroprev_target) else {}
    select_cases = cases_target.loc[(start_time < cases_target.index) & (cases_target.index < end_time)]
    select_deaths = deaths_target.loc[(start_time < deaths_target.index) & (deaths_target.index < end_time)]
    select_hosps = hosp_target.loc[(start_time < hosp_target.index) & (hosp_target.index < end_time)]   
    all_targets = {
        "weekly_cases": StandardDispTarget(select_cases, weight=20.0 * len(select_cases) / len(select_deaths)),
        "weekly_deaths": StandardDispTarget(select_deaths, weight=20.0),
        hosp_output: StandardDispTarget(select_hosps, weight=20.0 * len(select_hosps) / len(select_deaths)),
        "prop_eu": StandardDispTarget(prealpha_prop, weight=20.0),
    } | seroprev_target_dict

    # Seeding variants
    val = 0.5
    before_prop_time = (prealpha_prop - val).abs().idxmin() - timedelta(80)
    alpha_seed_start = max([before_prop_time, start_time])
    seed_times = [[alpha_seed_start, alpha_seed_start + timedelta(seed_duration)]]

    # Mobility
    mobility = get_country_mobility(iso3)

    for mob_analysis_type in mobility.columns[:1]:
        print(mob_analysis_type)
        
        # Define model and fitter
        model = MultiStrainModel(
            pop,
            start_time,
            end_time,
            proc_update_freq,
            CosineMultiCurve(),
            GammaDens(),
            init_duration,
            init_data,
            GammaDens(),
            GammaDens(),
            ["eu", "alpha"],
            "eu",
            seed_times,
            100.0,
            mobility[mob_analysis_type].dropna(),
        )
        
        # Run calibration
        calib = StandardCalib(model, priors, all_targets, proc_dispersion=dist.HalfNormal(0.5))
        kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
        mcmc = infer.MCMC(kernel, num_chains=4, num_samples=iterations, num_warmup=iterations)
        mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])
        store_outputs(country, mob_analysis_type, analysis_time, model, calib, mcmc)